# Automated Meeting Minutes Generator

Welcome to this professional guide on building an automated meeting minutes generator.

In this notebook, we will combine the power of **OpenAI's Whisper** model for audio transcription and **Meta's Llama 3.1** for text summarization to create a robust pipeline that converts raw audio into structured meeting minutes.

We will cover:
1.  **Environment Setup**: Configuring API keys and libraries.
2.  **Audio Transcription**: Using OpenAI's API to transcribe audio files.
3.  **Model Setup**: Loading a quantized Llama 3.1 model locally.
4.  **Minutes Generation**: Prompting the LLM to extract key information.
5.  **Execution**: Running the full pipeline.

## 1. Environment Setup

We need to install the necessary libraries for interacting with OpenAI and running Hugging Face models.

In [ ]:
%pip install openai torch transformers bitsandbytes accelerate python-dotenv fpdf

### Authentication

We will load our API keys from a `.env` file to keep them secure. You need an `OPENAI_API_KEY` for transcription and a `HUGGINGFACE_API_KEY` for downloading Llama 3.1.

In [1]:
%run ../../config/llm_settings.py

In [2]:
%run ../../utils/llm_functions.py

In [21]:
# %pip install requests torch bitsandbytes transformers sentencepiece accelerate
# %pip install -U bitsandbytes

from huggingface_hub import login
from transformers import AutoTokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
from IPython.display import Markdown, display, update_display
import torch

In [4]:
hf_token = HF_TOKEN

if hf_token and hf_token.startswith("Bearer "):
    hf_token = hf_token.replace("Bearer ", "", 1).strip()

if not hf_token:
    raise RuntimeError("HF_TOKEN no está configurado.")

print("HF_TOKEN loaded:", bool(hf_token))

# login(hf_token, add_to_git_credential=True)

HF_TOKEN loaded: True


## 2. Audio Transcription

We use OpenAI's Whisper model to transcribe the meeting audio. 

**Note**: If you have a file named `denver_extract.mp3` in your directory, the code below will transcribe it. Otherwise, we will use a sample transcript for demonstration purposes.

In [5]:
audio_filename = "../../inputs/denver_extract.mp3"

In [6]:
audio_filename

'../../inputs/denver_extract.mp3'

In [7]:
#Iniciar sesión en OpenAI usando Secrets en Colab

AUDIO_MODEL = "whisper-1"

In [8]:
# Usa el modelo Whisper OpenAI para convertir el audio en texto
# Si prefieres usar un modelo de código abierto, el estudiante de la clase
# Youssef ha contribuido con una versión de código abierto
# que he agregado al final de este Colab

audio_file = open(audio_filename, "rb")
transcription = openai_client.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
print(transcription)

and kind of the confluence of this whole idea of the confluence week, the merging of two rivers and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now and it's a very big issue. So that is the reason that the back of the logo is considered water. So let me see the creation of the logo here. So that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism and you'll hear a little bit more about our confluence week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous People's Day. So thank you. Thank you so much and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag of the United Sta

In [14]:
# Clean up memory before moving on
torch.cuda.empty_cache()

In [15]:
system_message = "Eres un asistente que produce actas de reuniones a partir de transcripciones, con resumen, puntos clave de discusión, conclusiones y elementos de acción con los propietarios, en formato Markdown. Aunque el audio original esté en inglés, quiero que me proporciones el acta en español."
user_prompt = f"A continuación, se incluye un extracto de la transcripción de una reunión del consejo de Denver. Escribe las actas traducidas al castellano en formato Markdown, incluyendo un resumen con los asistentes, la ubicación y la fecha; puntos de discusión; conclusiones; y elementos de acción con los propietarios.\n{transcription}"

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]

In [16]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [17]:
# Define model identifiers
LLAMA = "meta-llama/Llama-3.1-8B-Instruct"
PHI4 = "microsoft/Phi-3-mini-4k-instruct"
GEMMA3 = "google/gemma-3-1b-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
MIXTRAL = "mistralai/Mixtral-8x7B-Instruct-v0.1"

In [18]:
# 1. Prepare the input
tokenizer = AutoTokenizer.from_pretrained(GEMMA3)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer, skip_prompt=True)
model = AutoModelForCausalLM.from_pretrained(GEMMA3, device_map="auto", quantization_config=quant_config)
outputs = model.generate(**inputs, max_new_tokens=2000, streamer=streamer)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

<end_of_turn>


In [23]:
response = tokenizer.decode(outputs[0])

In [24]:
display(Markdown(response))

<bos><start_of_turn>user
Eres un asistente que produce actas de reuniones a partir de transcripciones, con resumen, puntos clave de discusión, conclusiones y elementos de acción con los propietarios, en formato Markdown. Aunque el audio original esté en inglés, quiero que me proporciones el acta en español.

A continuación, se incluye un extracto de la transcripción de una reunión del consejo de Denver. Escribe las actas traducidas al castellano en formato Markdown, incluyendo un resumen con los asistentes, la ubicación y la fecha; puntos de discusión; conclusiones; y elementos de acción con los propietarios.
and kind of the confluence of this whole idea of the confluence week, the merging of two rivers and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now and it's a very big issue. So that is the reason that the back of the logo is considered water. So let me see the creation of the logo here. So that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism and you'll hear a little bit more about our confluence week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous People's Day. So thank you. Thank you so much and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag of the United States of America, and to the republic for which it stands, one nation, under God, indivisible, with liberty and justice for all. All right. Thank you, Councilman Lopez. Madam Secretary, roll call. Clerk. Here. Espinosa. Here. Flynn. Here. Gilmour. Here. Here. Cashman. Here. Kenneche. Here. Lopez. Here. New. Here. Ortega. Here. Sussman. Here. Mr. President. Here. 11 present. 11 members present. We do have a quorum. Approval of the minutes. Seeing none. Minutes of October 2nd stand approved. Council announcements. Are there any announcements by members of Council? Councilman Clark. Thank you, Mr. President. I just wanted to invite everyone down to the first ever Halloween parade on Broadway in Lucky District 7. It will happen on Saturday, October 21st at 6 o'clock p.m. It will move along Broadway from 3rd to Alameda. It's going to be a fun, family-friendly event. Everyone's invited to come down, wear a costume. There will be candy for the kids and there are tiki zombies and 29 hearses and all kinds of fun and funky stuff on the fun and funky part of Broadway. So please join us October 21st at 6 o'clock for the Broadway Halloween parade. Thank you, Mr. President. All right. Thank you, Councilman Clark. I will be there. All right. Presentations. Madam Secretary, do we have any presentations? None, Mr. President. Communications. Do we have any communications? None, Mr. President. We do have one proclamation this evening. Proclamation 1127, an observance of the annual Indigenous Peoples Day in the City and County of Denver. Councilman Lopez, will you please read it? Thank you, Mr. President, with pride. Proclamation number 17, well, let me just say this differently. Proclamation number 1127, series of 2017, an observance of the second annual Indigenous Peoples Day in the City and County of Denver. Whereas the Council of the City and County of Denver recognizes that the Indigenous Peoples have lived and flourished on the lands known as the Americas since time immemorial and that Denver and the surrounding communities are built upon the ancestral homelands of numerous Indigenous tribes, which include the Southern Ute, the Ute Mountain, Ute tribes of Colorado. And whereas the tribal homelands and seasonal encampments of the Arapaho and Cheyenne people along the banks of the Cherry Creek and South Platte River confluence gave bearing to the future settlements that would become the birthplace of the Mile High City. And whereas Colorado encompasses the ancestral homelands of 48 tribes and the City and County of Denver and surrounding communities are home to the descendants of approximately 100 tribal nations. And whereas on October 3rd, 2016, the City and County of Denver unanimously passed Council Bill 801, series of 2016, officially designating the second Monday of October of each year as Indigenous Peoples Day in Denver, Colorado. And whereas the Council of the City and County of Denver continues to recognize and value the vast contributions made to the community through Indigenous Peoples' knowledge, science, philosophy, arts, and culture. And through these contributions, the City of Denver has developed and thrived. Whereas the Indigenous community, especially youth, have made great efforts this year to draw attention to the contributions of Indigenous people, including Confluence Week, drawing record attendance to a National Indigenous Youth Leadership Conference, leading conversations on inclusion with their peers, and supporting increased Indigenous youth participation in science and engineering. Now, therefore, be it proclaimed by the Council of the City and County of Denver, Section 1, that the Council of the City and County of Denver celebrates and honors the cultural and foundational contributions of Indigenous people to our history, our past, our present, and future, and continues to promote the education of the Denver community about these historical and contemporary contributions of Indigenous people. Section 2, that the City and County of Denver, Colorado, does hereby observe October 9th, 2017, as Indigenous Peoples Day. Section 3, that the Clerk of the City and County of Denver shall attest and affix the seal of the City and County of Denver to this proclamation, and that a copy be transmitted to the Denver American Indian Commission, the City and County of Denver School District No. 1, and the Colorado Commission on Indian Affairs. Thank you, Councilman Lopez. Your motion to adopt. Mr. President, I move that Proclamation No. 1127, Series of 2017, be adopted. All right. It has been moved and seconded. It comes from the members of Council. Councilman Lopez. Thank you, Mr. President. It gives me a lot of pleasure and pride to read this proclamation officially for the third time, but as Indigenous Peoples Day in Denver, officially for the second time. It's always awesome to be able to see not just this proclamation come by my desk, but to see so many different people from our community in our Council Chambers. It was a very beautiful piece of artwork that you presented to us earlier, and it is exactly the spirit that we drafted this proclamation and this actual, the ordinance that created Indigenous Peoples Day when we sat down and wrote it, and as a community, we couldn't think of anything else to begin except for the confluence of the two rivers. And those confluence of the two rivers created such a great city. And we live in such an amazing city, and we're all proud of it, and sometimes we, and a lot of people from all over the country or all over the world are proud of it, and sometimes a little too proud of it is telling them to go back home. But I'm kidding when I say that. But the really nice thing about this is that we are celebrating Indigenous Peoples Day out of pride for who we are, who we are as a city, and the contributions of Indigenous people to the city, not out of spite, not out of a replacement of one culture over the other, or out of contempt or disrespect. I think of a quote that Cesar Chavez made very popular, and it stuck with me for a very long time. And any time I have the opportunity to speak in front of children, and especially children in our community, that they often second guess themselves on where they're coming from and who they are. And I always say that it's very important to be proud of who you're from. And the quote that I use from Cesar Chavez is, you know, pride in one's own cultures does not require contempt or disrespect of another. Right? And that's very important. It's very important for us to recognize that. No matter who we are, where we come from in this society, your pride in your own culture does not require the contempt or disrespect of another. And man, what a year for that to just sit on our shoulders for a while, for us to think about. Right? And so I wanted to just to thank you all, to thank the commission. There's going to be a couple individuals that are going to come speak. Thank you for your art, your lovely artwork, for us to see what's in your heart and what now has become, probably is going to be a very important symbol for the community. And also just for the work, the daily work, every single day. We still have a lot of brothers and sisters whose ancestors once lived in these lands freely now stand on street corners. Right? In poverty. Without access to services. Right? Without access to sobriety or even housing or jobs. And what a cruel way to pay back a culture that has paved the way for the city to be built upon its shores. Right? So we have a lot of work to do. And these kind of proclamations and this day is not a day off, it's a day on in Denver. Right? And addressing those critical issues. So I know that my colleagues are very supportive. I'm going to ask you to support this proclamation as I know you always have done in the past. I'm very proud of today. Oh, and we made Time Magazine and Newsweek once again today as being a leader in terms of the cities that are celebrating Indigenous Peoples Day. I wanted to make a point out of that. Thank you, Councilman Lopez, and thank you for sponsoring this. Councilman Ortega? Mr. President, I want to ask that my name be added. I don't think I could add much more to what Councilman Lopez has shared with us. I want to thank him for bringing this forward and really just appreciate all the contributions that our Native American community has contributed to this great city and great state. I worked in the Lieutenant Governor's office when the Commission on Indian Affairs was created and had the benefit of being able to go down to the Four Corners for a peace treaty signing ceremony between the Utes and the Comanches that had been sort of at odds with each other for about a hundred years. And just being able to participate in that powwow was pretty awesome. And for those of you who continue to participate in the annual powwow, it's such a great opportunity for everybody else to enjoy so many of the contributions of the culture. I mean, to see that the dance continues to be carried on as well as the Native language from generation to generation is just so incredible because in so many cultures, you know, people have come here and assimilated to the, you know, the norms here and they lose their language and lose a lot of the culture. And in the Native community, that hasn't happened. That, you know, commitment to just passing that on from generation to generation is so important. And so I'm happy to be a co-sponsor of this tonight. Thank you. All right. Thank you, Councilwoman Ortega. Councilwoman Caniche. Thank you very much. And I also want to thank my colleague for bringing this forward. And I just wanted to say a word to the artist about how beautiful and moving I thought this logo was and your description of it. And I think one of the things that is clear is, you know, the words sometimes don't convey the power of imagery or music or the other pieces that make up culture. And so I think the art is so important. And when you talked about water, I was also thinking about land. And I guess I just wanted to say thank you. Many of the Native American peoples of Colorado have been at the forefront, or actually nationally, of defending some of the public lands that have been protected over the last few years that are under attack right now. And they're places that the communities have fought to protect but that everyone gets to enjoy. And so I just think that it's an example of where cultural preservation intersects with environmental protection, with, you know, recreation and all of the other ways that public lands are so important. And so I think I just wanted to say thank you for that because I think we have some very sacred places in our country that are at risk right now. And so as we celebrate, I appreciate that there's still a piece of resistance in here. And I think that I just want to mention a feeling of solidarity with that resistance. So thank you, and happy Confluence Week. Thank you, Councilwoman Kinneach. And seeing no other comments, I'll just say a couple. And in a time of such divisive ugliness and just despicable behavior from our leadership, the reason I'm so supportive of Indigenous Peoples' Day is because it means inclusivity. It means respecting all, respecting those who have been silenced on purpose for a long time and whose history has not been told. And so we celebrate inclusivity in the face of such evil times, honestly.<end_of_turn>
<end_of_turn>

## 3. Model Setup

We will load **Llama 3.1 8B Instruct** using 4-bit quantization to ensure it fits in memory while maintaining high performance.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TextStreamer

# Model Identifier
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

# Quantization Configuration
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# Load Model
print(f" Loading {MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    device_map="auto", 
    quantization_config=quant_config
)
print(f" Model loaded successfully")

## 4. Minutes Generation

Now we will construct a prompt that instructs the LLM to act as a professional secretary. We will feed it the transcript and ask for a structured output containing:
- Summary
- Discussion Points
- Takeaways
- Action Items

In [ ]:
# Define System and User Prompts
system_message = """
You are an expert administrative assistant. You produce high-quality minutes of meetings from transcripts.
Your output should be in Markdown format and include:
1. Summary (Attendees, Location, Date)
2. Key Discussion Points
3. Takeaways
4. Action Items with Owners
Do not use code blocks.
"""

user_prompt = f"""
Below is a transcript of a Denver City Council meeting.
Please write the minutes based on this text:

{transcription_text}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
]

# Prepare Inputs
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer, skip_prompt=True)

# Generate Minutes
print("\n--- Generating Minutes ---\n")
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

## 5. Interactive Interface with Gradio

Finally, we will create a simple user interface using **Gradio**. This allows users to:
1.  View and edit the transcription.
2.  Generate the meeting minutes.
3.  Download the result as a PDF file.

In [ ]:
import gradio as gr
from fpdf import FPDF
import tempfile
import os
from threading import Thread
from transformers import TextIteratorStreamer

def generate_minutes(transcription):
    """
    Genera las actas usando el LLM y transmite (stream) la salida a Gradio en tiempo real.
    """
    system_message = """
    You are an expert administrative assistant. You produce high-quality minutes of meetings from transcripts.
    Your output should be in Markdown format and include:
    1. Summary (Attendees, Location, Date)
    2. Key Discussion Points
    3. Takeaways
    4. Action Items with Owners
    Do not use code blocks.
    """
    
    user_prompt = f"""
    Below is a transcript of a Denver City Council meeting.
    Please write the minutes based on this text:
    
    {transcription}
    """
    
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]
    
    # Verificar que el modelo y el tokenizador estén cargados
    if 'model' not in globals() or 'tokenizer' not in globals():
        yield "Error: Model or tokenizer not loaded. Please run the previous cells."
        return

    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
    
    # Configurar el streamer para capturar el texto generado
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    generation_kwargs = dict(
        inputs=inputs, 
        streamer=streamer, 
        max_new_tokens=2000, 
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Ejecutar la generación en un hilo separado para no bloquear la interfaz
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()
    
    # Leer del streamer y enviar a Gradio poco a poco
    generated_text = ""
    for new_text in streamer:
        generated_text += new_text
        yield generated_text

def save_to_pdf(text):
    """
    Convierte el texto generado a un archivo PDF.
    """
    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.set_font("Arial", size=12)
    
    # Sanitización simple para compatibilidad con PDF (latin-1)
    sanitized_text = text.encode('latin-1', 'replace').decode('latin-1')
    
    for line in sanitized_text.split('\n'):
        pdf.multi_cell(0, 10, line)
        
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
    pdf.output(temp_file.name)
    return temp_file.name

# Crear la Interfaz de Gradio
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Automated Meeting Minutes Generator")
    gr.Markdown("Edit the transcription below if needed, then generate and download your minutes.")
    
    with gr.Row():
        with gr.Column():
            transcription_input = gr.Textbox(
                label="Audio Transcription", 
                value=transcription_text if 'transcription_text' in globals() else "", 
                lines=15
            )
            generate_btn = gr.Button("Generate Minutes", variant="primary")
        
        with gr.Column():
            minutes_output = gr.Markdown(label="Generated Minutes")
            download_btn = gr.Button("Download PDF")
            pdf_file = gr.File(label="Download")

    # Estado para guardar el texto final para el PDF
    minutes_state = gr.State("")

    # Manejadores de Eventos
    def on_generate(transcription):
        # La función ahora es un generador (usa yield)
        for partial_minutes in generate_minutes(transcription):
            yield partial_minutes, partial_minutes

    def on_download(minutes):
        if not minutes:
            return None
        return save_to_pdf(minutes)

    generate_btn.click(
        fn=on_generate, 
        inputs=[transcription_input], 
        outputs=[minutes_output, minutes_state]
    )
    
    download_btn.click(
        fn=on_download, 
        inputs=[minutes_state], 
        outputs=[pdf_file]
    )

# Lanzar la aplicación
print("Starting Gradio interface...")
demo.launch(share=True, debug=True)

## Conclusion

We have successfully built an automated pipeline that:
1.  Transcribes audio using state-of-the-art speech-to-text (Whisper).
2.  Processes the text using a powerful local LLM (Llama 3.1).
3.  Generates professional meeting minutes automatically.

This workflow can be adapted for various other use cases, such as summarizing lectures, interviews, or podcasts.